[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/06_ai_engineering/24_tools_and_agents.ipynb)

# 📓 Notebook 24 — Tool Calling and Small Agents

> **Module:** AI Engineering · **Estimated time:** 60–80 min · **Difficulty:** Intermediate

So far, an LLM has been a function that takes text and returns text. This notebook changes that. We give the model **tools** — functions it can decide to call — and watch it do multi-step work: look up data, do arithmetic, query a DataFrame, then summarise the result.

This single pattern is the backbone of **every modern AI product**: ChatGPT plugins, Cursor's code edits, Slack's AI summaries, every internal copilot. Once you understand the call → execute → return loop, you can build all of them.

> 🎯 **Our running example: a support-ops data assistant.** Everything below is built around one concrete goal — a small assistant that answers business questions about a customer-support dataset (*"how many tickets in total?"*, *"which channel is cheapest?"*) by running pandas queries on its own. By the end you'll have that ~100-line program, assembled one tool at a time.

> 🧭 **Mental model — planner + hands + a `while` loop.** Keep three sentences in your pocket for the whole notebook: the **model is the planner** (it only ever *asks* — emits text saying "call tool X"), **your code is the hands** (it *acts* — looks up the real function and runs it), and **the loop is the agent** (ask → run → tell the model the result → repeat). "Tool calling", "function calling", "agents", "ReAct" are all brand names for this one machine. We unpack it from scratch in Section 1 and call back to it the whole way down.

By the end you will have built a small **data assistant** — a 100-line program that answers business questions about the AI support-bot dataset by running pandas queries on its own.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Define a **tool** with a JSON schema the model can read.
2. Run the **call → execute → return** loop manually, step by step.
3. Build a multi-tool agent that picks the right tool for each question.
4. Add a **safety budget** (max steps, max tool calls) to prevent runaway loops.
5. Log every tool call so you can debug what the agent did.
6. Recognise when an agent is overkill — and when a single LLM call is enough.

## ✅ Prerequisites

Notebooks 22 (LLM workflows) and 23 (embeddings & retrieval). Familiarity with JSON schemas (NB 4) helps.

## 1. The tool-calling loop in one picture

```
   ┌───────────────────────────────────────────────────────────────┐
   │                                                                │
   │   user message                                                 │
   │       │                                                        │
   │       ▼                                                        │
   │   ┌───────┐                                                    │
   │   │  LLM  │── responds with: "call tool get_total_tickets()"   │
   │   └───┬───┘                                                    │
   │       │                                                        │
   │       ▼                                                        │
   │   your code looks up that tool name, runs it, captures result  │
   │       │                                                        │
   │       ▼                                                        │
   │   ┌───────┐                                                    │
   │   │  LLM  │── sees the result and: "now I can answer"          │
   │   └───┬───┘                                                    │
   │       │                                                        │
   │       ▼                                                        │
   │   final answer to user                                         │
   │                                                                │
   └────────────────────────────────────────────────────────────────┘
```

The "magic" is that the model gets to **decide** which tool to call and with what arguments. Your job is to:

1. Describe each tool to the model (name + JSON-schema arguments).
2. Listen for tool-call responses.
3. Run the matching Python function.
4. Send the result back to the model.
5. Loop until the model produces a final answer.

### 🔬 What actually happens when an LLM "uses a tool"?

Before we build the richer `MockLLM` below, let's slow the loop right down — because the single biggest misconception about agents is hiding here.

**An LLM cannot run code. It cannot fetch data. It cannot open a file.** A language model does exactly *one* thing: given some text, it produces more text. That's it. So when you hear "the model called the weather API" or "the agent ran a calculation", that is **never literally true**. What actually happened is a four-beat dance between two parties:

```text
   THE MODEL (the "planner")              YOUR CODE (the "hands")
   - can only EMIT TEXT                   - can actually DO things
   - decides WHAT to do                   - runs real Python functions
```

The model is the **brain that asks**; your code is the **hands that act**. "Tool use" is just a text protocol the two agree on:

1. **You describe the tools** to the model as text: *"You have a tool `add(a, b)`."*
2. **The model emits text that says "please call `add` with `a=2, b=3`."** It does *not* run anything — it just writes a request.
3. **Your code reads that request, looks up the real Python function, and runs it.** This is the only place real computation happens.
4. **You paste the result back into the conversation** as more text: *"`add` returned `5`."*
5. **The model reads that and continues** — maybe asking for another tool, maybe writing the final answer.

That observe → decide → act → observe cycle, repeated until the model is done, **is what we call an "agent".** Nothing more mystical than a `while` loop.


### The loop, drawn as a `while` loop

Here is the exact same picture from above, but annotated to show **who does what** at each beat. Keep your eye on the boundary line — text crosses it in both directions, but *real work only ever happens on the right.*

```text
        MODEL  (only emits text)        │        YOUR CODE  (does real work)
   ─────────────────────────────────────┼──────────────────────────────────────
                                         │
   user question  ─────────────────────▶│  put question into `messages`
                                         │        │
                                         │        ▼
            reads messages  ◀────────────│  fake_llm(messages)
                 │                        │
                 ▼                        │
   emits  {"tool":"add",                  │
           "args":{"a":2,"b":3}}  ──────▶│  is it a tool request?
                                         │     │ yes
                                         │     ▼
                                         │  fn = registry["add"]      ← LOOK UP
                                         │  result = fn(a=2, b=3)     ← RUN IT
                                         │     │
                                         │     ▼
            reads result  ◀──────────────│  append result to messages
                 │                        │        │
                 ▼                        │        └──▶ loop again ↺
   emits  {"answer": "It is 5."}  ──────▶│  is it a final answer?
                                         │     │ yes → STOP, return it
```

Read the boundary as a rule you can rely on forever: **everything the model produces is just text describing an intention.** Your code is what turns that intention into an action and reports back. Let's prove it with ~30 lines of stdlib Python — no API, no internet, fully deterministic.


In [ ]:
# 🧪 The whole mechanism, from scratch — stdlib only, fully offline.

# ── PART 1: the tool registry — a plain dict of {name: real python function} ──
# This is "the hands". These functions do actual work. The model never touches
# them directly; only OUR code below looks them up by name and calls them.
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

TOOL_REGISTRY = {
    "add":      add,
    "multiply": multiply,
}

# ── PART 2: the "model" — a MOCK that ONLY emits text (here: dicts) ──────────
# A real LLM would read `messages` and generate this. We hardcode a tiny
# deterministic script so you can see the exact shapes that cross the boundary.
# It returns EITHER a tool request  {"tool": ..., "args": {...}}
#            OR a final answer       {"answer": "..."}.
# Goal: compute (2 + 3) * 4, which needs TWO tool calls in sequence.
def fake_llm(messages):
    # Decide based on how many tool RESULTS we've already been shown.
    results = [m for m in messages if m["role"] == "tool"]
    if len(results) == 0:
        # Beat 1: no results yet → ask to add 2 + 3.
        return {"tool": "add", "args": {"a": 2, "b": 3}}
    elif len(results) == 1:
        # Beat 2: we were told add→5 → ask to multiply that by 4.
        five = results[-1]["content"]
        return {"tool": "multiply", "args": {"a": five, "b": 4}}
    else:
        # Beat 3: we have the product → write the final answer in plain text.
        product = results[-1]["content"]
        return {"answer": f"(2 + 3) * 4 = {product}"}

# ── PART 3: the LOOP — "the agent" — drives the cycle until a final answer ────
def run_loop(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    print(f"USER: {question}\n")
    for step in range(1, max_steps + 1):
        reply = fake_llm(messages)                       # ask the model (text in → text out)

        if "answer" in reply:                            # FINAL answer → stop
            print(f"[step {step}] MODEL answers  → {reply['answer']}")
            return reply["answer"]

        # Otherwise it's a tool REQUEST. OUR code does the real work:
        name, args = reply["tool"], reply["args"]
        print(f"[step {step}] MODEL asks       → call {name}({args})")
        fn = TOOL_REGISTRY[name]                         # look the function up by name
        result = fn(**args)                              # RUN the real python — here only
        print(f"[step {step}] YOUR CODE runs    → {name} returned {result}")

        # Feed the result back into the conversation so the model can continue.
        messages.append({"role": "assistant", "content": f"call {name}"})
        messages.append({"role": "tool", "name": name, "content": result})

    return "(gave up — hit max_steps without a final answer)"

print(run_loop("What is (2 + 3) * 4?"))


### Reading the trace — three beats, two tool calls

That printout *is* an agent. Walk the beats:

| Beat | Who acts | What happened |
|---|---|---|
| step 1 | **model** asks | "call `add(a=2, b=3)`" — just a dict, no math done yet |
| step 1 | **your code** runs | looks up `add`, runs `add(2, 3)` → `5`, appends result |
| step 2 | **model** asks | now sees `5`, asks "call `multiply(a=5, b=4)`" |
| step 2 | **your code** runs | runs `multiply(5, 4)` → `20`, appends result |
| step 3 | **model** answers | sees `20`, emits final text `"(2 + 3) * 4 = 20"` → loop stops |

Three things worth burning in:

- **The model never computed anything.** Every `+` and `*` ran inside *your* Python on the right of the boundary. The model only ever produced little dicts describing what it *wanted*.
- **The loop is the agent.** `run_loop` is ~15 lines. Swap `fake_llm` for a real API call and the structure is **identical** — that's exactly what the richer `MockLLM` (and any production framework) does under the hood.
- **`max_steps` is the seatbelt.** A model that keeps asking for tools and never answers would loop forever. The counter guarantees the loop always terminates. (You'll see this same guard on the real `run_agent` in Section 5.)

> ⚠️ **The classic beginner trap.** "The LLM ran my function" / "the agent queried the database." It didn't. The LLM emitted *text asking for it*; **your** code did the running. Keeping that boundary crisp is what lets you debug agents — when something goes wrong, you can always ask: *did the model ask for the wrong thing (a planning bug), or did my code mis-run it (an execution bug)?* They live on opposite sides of the line.


### 🧠 Mental model — planner + hands + a `while` loop

Hold these three sentences and every agent framework you ever meet becomes legible:

```text
   THE MODEL is the PLANNER  → it only ASKS (emits text: "call tool X with args Y")
   YOUR CODE is the HANDS    → it ACTS (looks up the real function and runs it)
   THE LOOP is the AGENT     → ask → run tool → tell the model the result → repeat,
                               until the model stops asking and gives a final answer.
```

That's the whole idea. "Tool calling", "function calling", "agents", "ReAct", "tool use" — different brand names, **identical machinery**: a model that requests, code that executes, and a loop that ferries text back and forth between them. The rest of this notebook just makes each piece sturdier: real JSON tool *schemas* (so the model knows what arguments are valid), a registry of useful data-lookup tools, error handling, and a trace you can inspect. But the heartbeat never changes from the loop you just ran above.

> 🎯 **Keep this anchor.** Whenever an agent confuses you, redraw the boundary line and label each message: *is this the model asking, or my code acting?* Ninety percent of "why did my agent do that?" questions dissolve the moment you put each step on the correct side.


## 2. Setup — a richer MockLLM that understands tools

Real APIs (OpenAI, Anthropic, etc.) have built-in tool-calling support. To stay offline and deterministic, we extend the `MockLLM` from NB 22 with a small "decision rule" that looks at the user message and decides whether to call a tool.

In production this would be replaced by **one line**: the real model just *does* this for you.

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from typing import Any, Callable

# ---------------------------------------------------------------------
# A tiny offline LLM-with-tools — the same MockLLM idea as in llm_providers.py
# (introduced in NB 17), extended here with tool-calling and vendored inline so
# the notebook stays self-contained.
#
# Real models (OpenAI, Anthropic, Gemini) interpret tool schemas natively.
# Here we simulate the decision with simple keyword rules so the notebook
# runs without internet or an API key. The CONTRACT is identical:
#   - chat() returns either {"text": str} OR
#                          {"tool_call": {"name": str, "arguments": dict}}.
# Plug in a real provider and only the inside of chat() changes.
# ---------------------------------------------------------------------

class MockLLM:
    def __init__(self):
        self.calls = 0   # for cost-accounting demos

    def chat(self, messages: list[dict],
             tools: list[dict] | None = None,
             **_) -> dict:
        self.calls += 1
        user_msg = next((m["content"] for m in reversed(messages)
                          if m["role"] == "user"), "").lower()
        # If we already saw a tool result, produce the final answer.
        had_tool_result = any(m["role"] == "tool" for m in messages)

        if not tools or had_tool_result:
            return {"text": self._compose_final(messages)}

        # Heuristic tool routing — easy and replaceable
        for tool in tools:
            name = tool["name"]
            if self._should_call(name, user_msg):
                args = self._infer_args(tool, user_msg)
                return {"tool_call": {"name": name, "arguments": args}}

        # No tool seemed to match → just answer in text.
        return {"text": "I'm not sure how to answer that with the tools I have."}

    # ----- internal helpers --------------------------------------------------
    @staticmethod
    def _should_call(tool_name: str, user_msg: str) -> bool:
        rules = {
            "total_tickets":       ["how many", "total", "tickets"],
            "mean_satisfaction":   ["satisfaction", "csat", "happy"],
            "channel_summary":     ["summary", "summarise", "summarize", "overview", "report"],
            "filter_channel":      ["chat", "email", "phone", "web form", "social"],
            "calculator":          ["+", "-", "*", "/", "calculate", "compute"],
        }
        keywords = rules.get(tool_name, [])
        return any(kw in user_msg for kw in keywords)

    @staticmethod
    def _infer_args(tool: dict, user_msg: str) -> dict:
        # Fish out a channel name if one of the recognised words appears
        for ch in ("Chat", "Email", "Phone", "Web Form", "Social"):
            if ch.lower() in user_msg:
                return {"channel": ch} if "channel" in tool["parameters"]["properties"] else {}
        # Fish out a math expression for the calculator
        if tool["name"] == "calculator":
            m = re.search(r"([\-\d\.\+\*\/\(\)\s]+)", user_msg)
            return {"expression": (m.group(0).strip() if m else "0")}
        return {}

    @staticmethod
    def _compose_final(messages: list[dict]) -> str:
        # Find the last tool result in the conversation and verbalise it
        tool_outputs = [m for m in messages if m["role"] == "tool"]
        if tool_outputs:
            last = tool_outputs[-1]
            return f"Based on the tool '{last['name']}', the answer is: {last['content']}."
        return "(no tools were used; I have no answer.)"


llm = MockLLM()
print("MockLLM (tool-aware) ready ✅")


> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. Once you want real intelligence in the answers, swap one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else.
> See the [LLM Providers Guide](./A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


> 🎯 **The contract.** Notice that `chat()` returns one of two shapes — either `{"text": ...}` (a final answer) or `{"tool_call": ...}` (a request for your code to run something). Every real tool-calling API in the wild uses this exact contract.

## 3. Tool schemas — what the model needs to know

A tool is described to the model with a tiny JSON schema. The schema serves three purposes:

1. **Tells the model the tool exists** (so it can choose to call it).
2. **Describes what the tool does** (the `description` field is what the model reads).
3. **Specifies the arguments** (so the model produces valid JSON).

In [ ]:
# Our toy support-ops dataset (same shape as the support_ops table in NB 13)
np.random.seed(42)
channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
support_ops = pd.DataFrame({
    "channel":          np.repeat(channels, 12),
    "month":            list(range(1, 13)) * 5,
    "tickets_total":    np.random.randint(1000, 8000, size=60),
    "automation_rate":  np.clip(np.random.normal(0.6, 0.15, 60), 0.1, 0.95),
    "satisfaction":     np.clip(np.random.normal(4.0, 0.2, 60), 1, 5),
    "cost_per_ticket":  np.random.uniform(0.3, 5.5, 60),
})

# A tiny "tool registry"
def total_tickets() -> int:
    """Sum the tickets_total column across all channels and months."""
    return int(support_ops["tickets_total"].sum())

def mean_satisfaction(channel: str = None) -> float:
    """Average satisfaction overall, or for a specific channel."""
    if channel:
        sub = support_ops[support_ops["channel"] == channel]
        if sub.empty:
            return float("nan")
        return float(sub["satisfaction"].mean())
    return float(support_ops["satisfaction"].mean())

def channel_summary(channel: str) -> dict:
    """Return key metrics for a single channel."""
    sub = support_ops[support_ops["channel"] == channel]
    return {
        "channel":         channel,
        "n_months":        len(sub),
        "total_tickets":   int(sub["tickets_total"].sum()),
        "mean_auto":       round(float(sub["automation_rate"].mean()), 3),
        "mean_csat":       round(float(sub["satisfaction"].mean()), 2),
        "mean_cost":       round(float(sub["cost_per_ticket"].mean()), 2),
    }

def calculator(expression: str) -> float:
    """Evaluate a basic arithmetic expression. Numeric / operators only."""
    if not re.fullmatch(r"[\d\.\+\-\*\/\(\)\s]+", expression):
        raise ValueError(f"Unsafe expression: {expression!r}")
    return eval(expression)   # safe: we already validated the chars


# The schema each tool advertises to the LLM
TOOLS = [
    {
        "name": "total_tickets",
        "description": "Return the total number of tickets across all channels and months.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "mean_satisfaction",
        "description": "Return the mean customer-satisfaction (CSAT) score, optionally for one channel.",
        "parameters": {
            "type": "object",
            "properties": {"channel": {"type": "string",
                                        "description": "Channel name, e.g. 'Chat'."}},
            "required": [],
        },
    },
    {
        "name": "channel_summary",
        "description": "Return a summary of key metrics for one channel.",
        "parameters": {
            "type": "object",
            "properties": {"channel": {"type": "string"}},
            "required": ["channel"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression (only digits and + - * / parentheses).",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
]
# Index from name → callable (your code's "router")
TOOL_FNS = {
    "total_tickets":     total_tickets,
    "mean_satisfaction": mean_satisfaction,
    "channel_summary":   channel_summary,
    "calculator":        calculator,
}
print(f"Registered {len(TOOL_FNS)} tools: {list(TOOL_FNS.keys())}")


> 💡 **Why JSON schemas?** Because the LLM needs to know what arguments to produce — and you need a way to validate the output. JSON Schema is the industry-standard format every major LLM API supports.

## 4. One round of tool calling — manually

Let's walk through the loop step by step.

In [ ]:
# Step 1: user asks a question
user_question = "What's the mean satisfaction for Chat?"

# Step 2: send messages + tools to the LLM
messages = [
    {"role": "system",
     "content": "You are a data assistant. Use the available tools to answer questions about support_ops."},
    {"role": "user", "content": user_question},
]
response = llm.chat(messages=messages, tools=TOOLS)
print("LLM response:", response)


In [ ]:
# Step 3: if the model wants a tool, run it
if "tool_call" in response:
    call = response["tool_call"]
    print(f"\nLLM requested: {call['name']}({call['arguments']})")
    fn = TOOL_FNS[call["name"]]
    result = fn(**call["arguments"])
    print(f"Tool returned: {result}")

    # Step 4: feed the result back to the LLM as a 'tool' message
    messages.append({"role": "assistant", "content": "",
                     "tool_call": call})
    messages.append({"role": "tool", "name": call["name"],
                     "content": json.dumps(result)})

    # Step 5: ask the LLM again — now it can produce a final answer
    final = llm.chat(messages=messages, tools=TOOLS)
    print(f"\nFinal answer: {final.get('text')}")


> 🎯 **Three actors in the loop:** the user (whose question kicks things off), the LLM (which decides *what* to do), and your code (which actually *does* the thing). Tool calling is the contract between the second and third.

## 5. Wrapping the loop in a function

The five lines we just wrote will be the body of every agent we build. Let's put them in a function with a **safety budget**.

In [ ]:
def run_agent(user_question: str,
              tools_schema: list[dict],
              tool_registry: dict,
              max_steps: int = 5,
              verbose: bool = True) -> dict:
    """Run the call → execute → return loop until the LLM stops asking for tools."""
    messages = [
        {"role": "system",
         "content": "You are a data assistant. Use the available tools to answer questions."},
        {"role": "user", "content": user_question},
    ]
    log = []

    for step in range(1, max_steps + 1):
        response = llm.chat(messages=messages, tools=tools_schema)

        if "text" in response and "tool_call" not in response:
            log.append({"step": step, "action": "final_answer",
                         "text": response["text"]})
            return {"answer": response["text"], "log": log,
                    "n_calls": llm.calls, "steps_used": step}

        call = response.get("tool_call")
        if not call:
            return {"answer": "(no answer)", "log": log, "n_calls": llm.calls,
                    "steps_used": step}

        # Execute the tool
        fn = tool_registry.get(call["name"])
        try:
            result = fn(**call["arguments"]) if fn else f"Unknown tool: {call['name']}"
        except Exception as e:
            result = f"Tool error: {type(e).__name__}: {e}"

        log.append({"step": step, "action": "tool_call",
                     "tool": call["name"], "arguments": call["arguments"],
                     "result": result})
        if verbose:
            print(f"  [step {step}] {call['name']}({call['arguments']}) → {str(result)[:120]}")

        # Append to the conversation so the LLM sees the result next turn
        messages.append({"role": "assistant", "content": "", "tool_call": call})
        messages.append({"role": "tool", "name": call["name"],
                          "content": json.dumps(result, default=str)})

    return {"answer": "(stopped: max_steps reached)", "log": log,
            "n_calls": llm.calls, "steps_used": max_steps}


# Try the agent on three different questions
for q in [
    "How many tickets in total?",
    "What is the mean satisfaction for Phone?",
    "Give me a channel summary for Chat.",
]:
    print(f"\n❓ {q}")
    out = run_agent(q, TOOLS, TOOL_FNS)
    print(f"💡 {out['answer']}")


> 💡 **`max_steps` is the most important parameter on an agent.** Without it, a model that keeps requesting tools (or a tool that produces results the model can't interpret) will loop forever. Budget defensively.

## 6. Inspecting the trace — debugging an agent

When the data assistant gives a surprising answer, you don't guess — you read the tape. Production agents are debugged by reading their trace: the ordered list of *what the model decided*, *which tool ran*, and *what came back*. (Remember the boundary line from Section 1: the trace lets you see exactly which side a bug lives on — a planning mistake by the model, or an execution mistake in your hands.) Our function returns the trace as `out["log"]`.

In [ ]:
out = run_agent("Tell me the summary for Email", TOOLS, TOOL_FNS, verbose=False)
print("Trace:")
for entry in out["log"]:
    print(f"  {entry}")
print(f"\nTotal LLM calls: {out['n_calls']}    Steps used: {out['steps_used']}")


## 7. A multi-step agent — calculator chain

What if a question needs *two* tool calls? Let's give the agent a request that combines numerical lookup with arithmetic.

(The MockLLM here is simple, so this is a demonstration. With a real LLM you'd see it reason and chain naturally.)

> ⚠️ **Caveat — no real chaining offline.** The offline `MockLLM` short-circuits to a final answer as soon as it sees one tool result, so you won't see two tool calls chained here. True multi-step chaining (tool → tool → answer) requires a live LLM.

In [ ]:
# A tool that returns a number we can then feed into the calculator
out = run_agent("Compute 250 * 4", TOOLS, TOOL_FNS)
print(f"\nFinal: {out['answer']}")
print(f"Tool history: {[e.get('tool') for e in out['log'] if e['action']=='tool_call']}")


## 8. When NOT to use an agent

The planner + hands + loop is powerful — which makes it tempting to reach for it everywhere. Resist. Agents are powerful but expensive (multiple LLM calls per question, slow, harder to test). Don't reach for one when:

| Situation | Better choice |
|---|---|
| A single function would answer the user | Just expose the function, no LLM. |
| The user's question can be answered with one LLM call | Use a normal prompt. |
| The result must be exactly reproducible across runs | Code, not an LLM. |
| Latency must be under ~500 ms | Single call, not an agent loop. |

> 🎯 **A useful test.** If you can write the "if-this-then-that" logic in ~30 lines of Python, *write the 30 lines*. The LLM agent is for cases where the *decision tree* is too big or too fuzzy to enumerate.

## 9. Going live — the real-provider sketch

A real OpenAI / Anthropic tool-calling loop is almost identical to ours. Reference code:

```python
from openai import OpenAI
client = OpenAI()

def real_run(question, tools, tool_registry, max_steps=5):
    messages = [{"role": "system",  "content": "You are a data assistant."},
                {"role": "user",    "content": question}]
    for _ in range(max_steps):
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=[{"type": "function", "function": t} for t in tools],
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content                              # final answer
        for tc in msg.tool_calls:
            fn = tool_registry[tc.function.name]
            args = json.loads(tc.function.arguments)
            result = fn(**args)
            messages.append({"role": "assistant", "tool_calls": [tc.model_dump()]})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                              "content": json.dumps(result, default=str)})
    return "(max steps reached)"
```

Notice the shape: **same loop, same registry pattern, same safety budget.** Only the inside of the `client.chat.completions.create(...)` call differs. The investment you made in the mock pays off — your application code stays unchanged.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a tool

Add a tool `cheapest_channel()` that returns the channel with the lowest `mean cost_per_ticket`. Register it in `TOOL_FNS`, add its schema to `TOOLS`, and update the MockLLM's keyword rule so that questions like *"which channel is cheapest?"* route to it.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def cheapest_channel() -> dict:
    g = support_ops.groupby("channel")["cost_per_ticket"].mean()
    name = g.idxmin()
    return {"channel": name, "mean_cost": round(float(g.min()), 2)}

TOOL_FNS["cheapest_channel"] = cheapest_channel
TOOLS.append({
    "name": "cheapest_channel",
    "description": "Return the channel with the lowest mean cost per ticket.",
    "parameters": {"type": "object", "properties": {}, "required": []},
})

# The offline MockLLM routes by keyword, so it won't auto-discover this new tool.
# With a REAL provider you do nothing extra — the model reads the tool's
# "description" and decides to call it. To see it work offline, call it directly:
print(cheapest_channel())
```

In the MockLLM's `_should_call`, add `"cheapest_channel": ["cheap", "lowest cost", "cost"]`. With a real LLM the rule disappears — the model reads the new tool's `description` and routes on its own.
</details>

### Exercise 2 — ⭐⭐ Pretty-print the trace

Write `print_trace(out)` that displays the agent log as a numbered list, with one line per step showing either the tool call or the final answer.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def print_trace(out):
    for entry in out["log"]:
        n = entry["step"]
        if entry["action"] == "tool_call":
            print(f"  {n}. CALL {entry['tool']}({entry['arguments']})")
            print(f"     → {str(entry['result'])[:100]}")
        else:
            print(f"  {n}. ANSWER: {entry['text']}")

out = run_agent("Mean satisfaction for Chat", TOOLS, TOOL_FNS, verbose=False)
print_trace(out)
```

A clean trace printer is the single best agent-debugging tool. Production teams build dashboards out of these traces.
</details>

### Exercise 3 — ⭐⭐ Add an out-of-tools guard

Modify `run_agent` so that if the model requests a tool that **isn't registered**, the function returns a clear error message instead of crashing. (Hint: the function already handles unknown tools partially — make sure it both reports cleanly *and* stops the loop.)

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def run_agent_safe(user_question, tools_schema, tool_registry, max_steps=5):
    messages = [{"role": "system",
                 "content": "You are a data assistant."},
                {"role": "user", "content": user_question}]
    log = []
    for step in range(1, max_steps + 1):
        resp = llm.chat(messages=messages, tools=tools_schema)
        if "text" in resp and "tool_call" not in resp:
            log.append({"step": step, "action": "final_answer", "text": resp["text"]})
            return {"answer": resp["text"], "log": log}

        call = resp.get("tool_call")
        if not call:
            return {"answer": "(no answer)", "log": log}
        fn = tool_registry.get(call["name"])
        if fn is None:
            log.append({"step": step, "action": "tool_missing",
                         "tool": call["name"]})
            return {"answer": f"Error: model asked for unknown tool {call['name']!r}.",
                    "log": log}
        result = fn(**call["arguments"])
        log.append({"step": step, "action": "tool_call",
                     "tool": call["name"], "arguments": call["arguments"],
                     "result": result})
        messages.append({"role": "assistant", "content": "", "tool_call": call})
        messages.append({"role": "tool", "name": call["name"],
                          "content": json.dumps(result, default=str)})
    return {"answer": "(stopped)", "log": log}
```

**Why this matters.** Production LLMs occasionally hallucinate tool names. Your code should fail clearly when that happens — *stopping the loop with an explicit error* is much better than *silently returning garbage*.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The function below tries to handle multiple tool calls per response — but it has an off-by-one bug that causes it to skip every other call. Find it.

```python
def buggy_handle(calls, registry):
    results = []
    i = 0
    while i < len(calls):
        result = registry[calls[i]["name"]](**calls[i]["arguments"])
        results.append(result)
        i = i + 2     # ← bug!
    return results
```

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def handle(calls, registry):
    return [registry[c["name"]](**c["arguments"]) for c in calls]
```

The original was incrementing `i` by 2 each iteration, skipping every other call. The fix is `i += 1` (or, more Pythonically, just use a `for` loop / comprehension). This is the classic manual-index bug — Python rewards `for` loops (and comprehensions) over hand-managed counters.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ Add a `recent_low_csat_channels` tool

Add a new tool that returns the **top-3 channels with the lowest mean satisfaction**. Register it in `TOOL_FNS` and `TOOLS`, and prompt the agent with: *'which channels are unhappy?'*.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def recent_low_csat_channels(n: int = 3) -> list[dict]:
    means = support_ops.groupby("channel")["satisfaction"].mean().sort_values()
    return [{"channel": ch, "mean_csat": round(float(v), 2)}
            for ch, v in means.head(n).items()]


TOOL_FNS["recent_low_csat_channels"] = recent_low_csat_channels
TOOLS.append({
    "name": "recent_low_csat_channels",
    "description": "Return the top-n channels with the lowest mean customer-satisfaction score.",
    "parameters": {"type": "object",
                    "properties": {"n": {"type": "integer", "default": 3}},
                    "required": []},
})

# The offline mock routes by keyword and won't auto-discover new tools.
# With a real provider, registering the tool above is enough — the model reads
# its "description" and calls it when relevant. Offline, call it directly:
print(recent_low_csat_channels())
```

**The agent gets smarter for free.** Once the tool is registered,
a real LLM will choose to call it on any question that mentions
low CSAT — no further prompt engineering needed.

</details>

### Stretch exercise B — ⭐⭐⭐ Tool-execution timeout

Modify `run_agent` so that any single tool call must complete within `tool_timeout` seconds, or the agent logs an error and continues. Use `concurrent.futures.ThreadPoolExecutor` with a 2-second timeout.


In [ ]:
# Your code here  👇
from concurrent.futures import ThreadPoolExecutor, TimeoutError


<details>
<summary>💡 <b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor, TimeoutError

def run_agent_with_timeout(user_question, tools_schema, tool_registry,
                            max_steps=5, tool_timeout=2.0):
    messages = [{"role": "system", "content": "You are a data assistant."},
                {"role": "user",   "content": user_question}]
    log = []
    for step in range(1, max_steps + 1):
        resp = llm.chat(messages=messages, tools=tools_schema)
        if "text" in resp and "tool_call" not in resp:
            return resp["text"], log
        call = resp["tool_call"]
        fn   = tool_registry[call["name"]]
        try:
            with ThreadPoolExecutor(max_workers=1) as ex:
                fut = ex.submit(fn, **call["arguments"])
                result = fut.result(timeout=tool_timeout)
        except TimeoutError:
            result = f"<tool '{call['name']}' timed out after {tool_timeout}s>"
        log.append({"step": step, "tool": call["name"], "result": result})
        messages.append({"role": "tool", "name": call["name"],
                          "content": json.dumps(result, default=str)})
    return "(stopped — max_steps reached)", log


ans, log = run_agent_with_timeout("Mean satisfaction for Chat", TOOLS, TOOL_FNS)
print("Answer:", ans)
```

**Production tools fail.** A blocked database query, a stuck network
read, a runaway loop — any can hang your agent indefinitely.
A per-tool timeout (1-5 s) is the cheapest survival pattern.

</details>

### Stretch exercise C — ⭐⭐⭐ Two-tool agent: calculator + lookup

Extend the agent loop with **two** tools:

- `add(a, b)`
- `lookup_price(product)` — returns a dict like `{"price": 99.0}` for known products.

Given the user query *'How much do an apple and a banana cost together?'* the agent should call `lookup_price("apple")`, then `lookup_price("banana")`, then `add(...)` with the two prices, and finally print the total.

For this offline exercise you can call the tools by hand — just demonstrate the routing pattern.

In [ ]:
# Your code here  👇
PRICES = {"apple": 1.0, "banana": 0.5, "cherry": 3.0}

def add(a, b):              return a + b
def lookup_price(product):  return {"price": PRICES[product]}

TOOL_MAP = {"add": add, "lookup_price": lookup_price}

# Mock a 3-step agent trace as a list of (tool_name, args) tuples.
# Then execute them in order and print the result.
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
PRICES = {"apple": 1.0, "banana": 0.5, "cherry": 3.0}

def add(a, b):              return a + b
def lookup_price(product):  return {"price": PRICES[product]}

TOOL_MAP = {"add": add, "lookup_price": lookup_price}

trace = [
    ("lookup_price", {"product": "apple"}),
    ("lookup_price", {"product": "banana"}),
    ("add",          "__use_previous_two_prices__"),
]

memory = []
for name, args in trace:
    if name == "add":
        a, b = memory[-2]["price"], memory[-1]["price"]
        result = TOOL_MAP[name](a, b)
    else:
        result = TOOL_MAP[name](**args)
    memory.append(result)
    print(f"  → {name}({args})  ⇒  {result}")

print(f"\nTotal: ${memory[-1]:.2f}")
```

**Reasoning.** The agent loop's hardest design question isn't *which tools to expose* — it's **how the agent passes results between steps**. Three options you'll see in real frameworks. (1) **A scratchpad / memory list** (what we did) — simple, transparent, and the way ReAct-style agents work. (2) **Named variables** — the model says 'store this as `apple_price`', then references the name later (LangChain agents work this way). (3) **Implicit composition** via the LLM rewriting earlier results into the next prompt. Most production agents use a combination. The mock trace here makes the dataflow explicit — that's worth doing before you rely on an LLM to do it right.
</details>

### Stretch exercise D — ⭐⭐⭐ Add a 'refuse' tool

Production agents need a way to **decline** when the user asks for something they shouldn't do. Add a third tool `refuse(reason)` that the agent uses when a query is out of scope (e.g. medical advice from a customer-support bot).

Demonstrate the agent routing the query *'should I take ibuprofen for my headache?'* to `refuse` rather than to the LLM.

In [ ]:
# Your code here  👇
def lookup_price(product): return {"price": 1.0}
def add(a, b):             return a + b
def refuse(reason):        return {"refused": True, "reason": reason}

TOOL_MAP = {"lookup_price": lookup_price, "add": add, "refuse": refuse}

OUT_OF_SCOPE = {"medical", "legal", "financial advice", "diagnose"}

def route(query):
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def lookup_price(product): return {"price": 1.0}
def add(a, b):             return a + b
def refuse(reason):        return {"refused": True, "reason": reason}

TOOL_MAP = {"lookup_price": lookup_price, "add": add, "refuse": refuse}

OUT_OF_SCOPE = {"ibuprofen", "medical", "diagnose", "prescription", "legal", "lawyer"}

def route(query):
    q = query.lower()
    if any(w in q for w in OUT_OF_SCOPE):
        return TOOL_MAP["refuse"](
            "This bot only handles product/billing questions — please ask a qualified professional."
        )
    if "price" in q or "cost" in q:
        return TOOL_MAP["lookup_price"](product="apple")
    return TOOL_MAP["refuse"]("I'm not sure how to handle that.")

print(route("should I take ibuprofen for my headache?"))
print(route("what's the price of apples?"))
```

**Reasoning.** Refusal is a first-class tool, not an exception. Three reasons. (1) **Auditability** — every refusal is logged with a structured reason, which is what your compliance team is going to ask for. (2) **Consistency** — the agent always returns the same shape (`{refused: bool, ...}`), so downstream code doesn't branch on string content. (3) **Testability** — you can unit-test the routing rules without ever calling the LLM. In a real system you'd also have the LLM itself produce a `refuse(reason)` call when *it* doesn't want to comply, so the routing logic and the model's safety judgement converge on the same structured output.
</details>

## 🎁 Bonus mini-project — A pandas-query tool

Add a tool `run_pandas_query(expression: str)` that takes a pandas-style expression like `"support_ops.groupby('channel')['cost_per_ticket'].mean()"` and returns the result. **Validate the expression** (only allow operations on `support_ops`) before running it. Then ask the agent things like *"average cost per ticket per channel"*.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def run_pandas_query(expression: str) -> str:
    if "support_ops" not in expression:
        raise ValueError("Expression must reference support_ops only")
    # Whitelist of safe characters/keywords (very rough — production needs real sandboxing)
    if any(bad in expression for bad in ["import", "open(", "eval(", "exec(", "__"]):
        raise ValueError("Disallowed token in expression")
    result = eval(expression, {"support_ops": support_ops, "pd": pd, "np": np})
    return str(result)

TOOL_FNS["run_pandas_query"] = run_pandas_query
TOOLS.append({
    "name": "run_pandas_query",
    "description": "Run a pandas expression on support_ops. ONLY this DataFrame is accessible.",
    "parameters": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
})
```

**Why this matters.** A pandas-query tool turns a basic LLM into a **data analyst** — given the schema, it can answer any analytic question without you anticipating it. Most enterprise "data copilots" are this single tool plus a little prompt engineering.

**Safety warning.** `eval` is dangerous. Production implementations either (a) validate the expression with an AST whitelist, (b) run it in a subprocess with a tight resource budget, or (c) translate to SQL and execute against a read-only database. The validation above is *illustrative*, not production-ready.
</details>

## 🧠 Key takeaways

You set out to build a support-ops data assistant, and you got there by making one loop sturdier — schemas, a tool registry, error handling, a budget, a trace. The heartbeat never changed from the planner + hands + `while` loop you ran in Section 1:

1. **Tool calling = the LLM picks; your code runs.** The LLM never executes anything itself.
2. Tools are described with a **JSON schema** — name, description, argument types.
3. The agent loop is **5 lines**: call LLM → got tool_call? → run tool → append result → loop.
4. **Always set `max_steps`.** An agent without a budget is a kernel waiting to hang.
5. **Log every call.** The trace is your only debugging tool when an agent misbehaves.
6. **When unsure, code it.** Agents shine when the decision tree is too big to write by hand.
7. The shape of every tool-calling API (OpenAI / Anthropic / etc.) is identical — your code is portable.
8. A single `run_pandas_query` or `run_sql` tool turns an LLM into a real **data assistant**.

> 🧭 **The anchor, one last time.** Whenever an agent confuses you — yours or a framework's — redraw the boundary line: *is this message the model asking, or my code acting?* Planner emits text, hands do the work, the loop ferries results between them. Ninety percent of "why did my agent do that?" dissolves the moment each step lands on the correct side.

## ✅ Self-assessment

- [ ] Write a JSON-schema tool definition that a model could call
- [ ] Run the call → execute → return loop manually
- [ ] Build a `run_agent` function with `max_steps` and a tool registry
- [ ] Read an agent trace and identify which tool ran with which arguments
- [ ] Handle the case where the model requests an unknown tool
- [ ] Explain when an agent is worth it versus a single prompt

## 🚀 Next step

Continue with **Notebook 25 — AI Document Processing**, where the LLM stops just *answering* and starts *reading* — extracting structured fields from messy PDFs and invoices.